# NLP-04 | Verified Value-Adding Experiments — Phase 2
**Jigsaw multi-label toxic comments · Designed specifically for your existing Colab project**

### What this notebook adds (without repeating old transformer training)

| Study | Actual research question | Compute | Novel evidence |
|---|---|---|---|
| **A — Reproducibility + split audit** | Are our saved inputs and model scores really aligned? | CPU, seconds | Dataset-integrity and prediction-integrity report |
| **B — Lexical + transformer fusion** | Does adding your saved Logistic Regression or calibrated SVM *actually* improve the frozen RoBERTa–DistilBERT ensemble? | CPU, 1–3 min | Same-comment, paired comparison, per-label and uncertainty analysis |
| **C — Identity false positives + length stress** | Does any fusion change false alarms on benign identity-term comments or longer texts? | CPU, seconds | Side-by-side subgroup/robustness diagnostics |
| **D — Controlled training-data ablation** | Is the previous 34K toxicity-enriched training sample better or worse than a **representative 34K** and the **full 111K** on the same evaluation records? | CPU, several minutes | Separates some effects of size, prevalence, class weighting |

**Existing files used:** `/data/processed/train_sub.csv`, `train_full.csv`, `val.csv`; `/results/predictions/{A1,A2,R1_128,B1_256}_val.npy`; `/results/final/final_val.npy`.

**Important safeguards**
1. **No original dataset upload is needed.** Use your existing Google Drive project. This notebook does **not** change your previous models, splits, notebooks, predictions or `final_config.json`.
2. **Never reads official/internal test labels or predictions for model selection.** All new comparisons are exploratory on the existing development validation split. Because old models were already chosen using this validation set, the new half-validation audit is **not a genuinely unseen final test**.
3. Fusions are evaluated **with the same six labels, same row order, same 50/50 development split**, and the same per-label threshold procedure. SVM scores `A2` are **already sigmoid-calibrated probabilities**; raw uncalibrated LinearSVC decision scores must not be averaged with transformer probabilities.
4. A negative result is valuable: **do not claim improvement** if added models provide only a tiny or uncertain gain.
5. Long transformer retraining, extra CNN/LSTMs and fairness mitigation training are intentionally not included: these require more time, GPU, and a separate controlled protocol.

**Run the notebook top to bottom.** Studies A–C need no GPU. Study D uses CPU; a Colab High-RAM runtime helps but is not required if memory is sufficient.

In [ ]:
# CELL 1 — Project configuration, dependencies and safe output folder
import os, sys, json, time, re, hashlib, warnings
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score

PROJECT_DIR = Path(os.environ.get(
    'NLP04_PROJECT_DIR',
    '/content/drive/MyDrive/Colab Notebooks/jigsaw-toxic-classifier'
))
if not PROJECT_DIR.exists() and Path('/content').exists():
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        pass
assert PROJECT_DIR.exists(), (
    f'Existing project not found at: {PROJECT_DIR}. Please mount the Drive containing '
    'your original project, or set NLP04_PROJECT_DIR to that project directory. '
    'Do not create a new split.'
)
LABELS = ['toxic','severe_toxic','obscene','threat','insult','identity_hate']
RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
OUT = PROJECT_DIR / 'results' / f'value_add_phase2_{RUN_TAG}'
OUT.mkdir(parents=True, exist_ok=False)
SEED = 1729                       # Different from original training seed 42
BOOTSTRAP_REPS = 150             # Increase to 500 later if desired
RUN_SAMPLING_STUDY = True        # Turn False for the fast A–C studies only
RUN_BALANCED_VARIANT = False     # Optional fourth model; usually not needed for the 70% milestone
SAVE_NEW_LEXICAL_MODELS = False   # Save CSV/NPY results fast; set True later to retain model weights
print('Existing project:', PROJECT_DIR)
print('New outputs (original files untouched):', OUT)
print('Python/scikit-learn:', sys.version.split()[0], __import__('sklearn').__version__)

In [ ]:
# CELL 2 — Load saved partitions and perform strict integrity checks
PROC = PROJECT_DIR / 'data' / 'processed'
PRED = PROJECT_DIR / 'results' / 'predictions'
FINAL = PROJECT_DIR / 'results' / 'final'
need = [PROC/f'{s}.csv' for s in ['train_full','train_sub','val']]
need += [PRED/f'{r}_val.npy' for r in ['A1','A2','R1_128','B1_256']]
need += [FINAL/'final_val.npy', FINAL/'final_config.json']
missing = [str(x) for x in need if not x.exists()]
assert not missing, ('Required original saved project artifacts are missing. Run your old '
                     f'notebooks / restore your original Drive project first. Missing:\n'+'\n'.join(missing))
tr = pd.read_csv(PROC/'train_full.csv')
sub = pd.read_csv(PROC/'train_sub.csv')
val = pd.read_csv(PROC/'val.csv')
for name,df in [('train_full',tr),('train_sub',sub),('val',val)]:
    assert set(['id','comment_text'] + LABELS).issubset(df.columns), name+' has wrong columns'
    assert df.id.is_unique and not df.id.isna().any(), name+' has duplicate/missing IDs'
    assert not df[LABELS].isna().any().any(), name+' has missing labels'
    assert np.isin(df[LABELS].to_numpy(), [0,1]).all(), name+' must contain binary labels'
assert set(sub.id).issubset(set(tr.id)), 'Original train_sub must be a subset of train_full'
assert set(tr.id).isdisjoint(set(val.id)), 'FATAL: training IDs overlap validation IDs'

def normalizer(s):
    return s.fillna('').astype(str).str.lower().str.replace(r'\s+', ' ', regex=True).str.strip()
train_keys = set(normalizer(tr.comment_text))
overlap_text = len(train_keys.intersection(set(normalizer(val.comment_text))))
assert overlap_text == 0, f'FATAL: {overlap_text} normalized-text overlaps train vs val'
assert len(tr)==111707 and len(sub)==34044 and len(val)==23932, (
    'Unexpected split row counts. Verify you are using the exact original project artifacts.'
)
Ytr = tr[LABELS].to_numpy(dtype=np.int8)
Yv = val[LABELS].to_numpy(dtype=np.int8)
print('Verified: train_full',len(tr),'| train_sub',len(sub),'| val',len(val))
print('Train ↔ val normalized comment overlap:', overlap_text)
print('Validation label positives:', dict(zip(LABELS,Yv.sum(axis=0).tolist())))

# Aligning .npy arrays by row position is only valid if the original generation used this val.csv order.
# Reproduce historical aggregate AP numbers as a strong sanity check (not proof of per-row ID alignment).
original = {
    'FROZEN_2T': np.load(FINAL/'final_val.npy'),
    'R1_RoBERTa': np.load(PRED/'R1_128_val.npy'),
    'B1_DistilBERT': np.load(PRED/'B1_256_val.npy'),
    'A1_LR': np.load(PRED/'A1_val.npy'),
    'A2_CalibratedSVM': np.load(PRED/'A2_val.npy'),
}
for name,p in original.items():
    assert p.shape == (len(val),6), (name,p.shape,'wrong shape')
    assert np.isfinite(p).all(), name+' contains nonfinite values'
    assert ((p >= 0) & (p <= 1)).all(), name+' is NOT a probability matrix'
known = {'FROZEN_2T':0.7551, 'R1_RoBERTa':0.7403, 'B1_DistilBERT':0.7307,
         'A1_LR':0.6831, 'A2_CalibratedSVM':0.6715}
for name,p in original.items():
    ap=float(average_precision_score(Yv,p,average='macro'))
    assert abs(ap-known[name]) < 0.004, f'{name}: historical AP mismatch {ap:.5f}; inspect artifact version/order'
    print(f'PASS {name:19s} validation macro AP {ap:.5f}')

fingerprints = {}
for nm, df in [('train_full',tr),('train_sub',sub),('val',val)]:
    fingerprints[nm] = hashlib.sha256('\n'.join(df.id.astype(str)).encode()).hexdigest()
metadata = {
    'project':str(PROJECT_DIR), 'output':str(OUT), 'seed':SEED,
    'label_order':LABELS, 'split_sizes':{'train_full':len(tr),'train_sub':len(sub),'val':len(val)},
    'ordered_id_sha256':fingerprints, 'known_ap_check':'passed',
    'limitations':['No genuinely untouched test: original validation already used in previous model development',
                   'Historical prediction arrays have no row-wise ID metadata; score checks support but do not prove alignment']
}
(OUT/'run_manifest.json').write_text(json.dumps(metadata,indent=2))

## Study B — Fusion and threshold robustness, **development data only**
We will use a **50% threshold/model-selection portion** and a distinct **50% reporting portion** of your original validation set (six-label-combination stratification, pooling extremely rare combinations). Candidate fusion weights are **predeclared** below. Every model gets thresholds tuned on the **same selection portion**; no thresholds are fit to the reporting portion.

**Crucial caveat:** Some underlying models were selected previously using the entire validation set. This retrospective reporting split is therefore not a new independent test. It is a valid diagnostic, **not** evidence of previously unseen generalization.

In [ ]:
# CELL 3 — Define standardized scoring and a fixed multi-label validation split
# Stratify on the six-label combination (0..63); merge exceedingly rare combinations.
# Unlike splitting by 'any toxic', this protects common co-occurring label patterns.
def stratified_multilabel_indices(y, train_size, seed):
    codes = (y.astype(np.int32) * (2 ** np.arange(y.shape[1]))).sum(axis=1)
    counts = np.bincount(codes, minlength=2**y.shape[1])
    strata = np.where(counts[codes]>=2, codes, -1)
    splitter = StratifiedShuffleSplit(n_splits=1, train_size=train_size, random_state=seed)
    a,b=next(splitter.split(np.zeros((len(y),1)), strata))
    return a,b
idx_sel, idx_audit = stratified_multilabel_indices(Yv, train_size=len(val)//2, seed=SEED)
assert len(set(idx_sel).intersection(idx_audit))==0
assert len(idx_sel)+len(idx_audit)==len(val)
print('Selection rows:',len(idx_sel),'| Retrospective audit rows:',len(idx_audit))
print('Audit positives:',dict(zip(LABELS,Yv[idx_audit].sum(axis=0).tolist())))

# Match your original toxlib threshold grid: 0.05, 0.06, ... 0.95
THR_GRID = np.round(np.arange(0.05, 0.951, 0.01),2)

def select_thresholds(Y, P):
    """Per-label F1-maximizing thresholds on selection data ONLY. Stable tie: first grid point."""
    out = np.zeros(6, dtype=float)
    for j in range(6):
        truth = Y[:,j].astype(bool)
        flags = P[:,j,None] >= THR_GRID[None,:]
        tp = np.sum(flags & truth[:,None], axis=0)
        fp = np.sum(flags & ~truth[:,None], axis=0)
        fn = np.sum(~flags & truth[:,None], axis=0)
        f1 = np.divide(2*tp,2*tp+fp+fn,out=np.zeros_like(tp,dtype=float),where=(2*tp+fp+fn)>0)
        out[j] = THR_GRID[int(np.argmax(f1))]
    return out

def metrics(Y,P, thresholds):
    """Macro AP, macro/micro F1 and six-label metrics on the SAME Y and scores."""
    assert P.shape == Y.shape and P.shape[1]==len(LABELS)
    pred=(P >= np.asarray(thresholds)[None,:]).astype(np.int8)
    return dict(macro_ap=float(average_precision_score(Y,P,average='macro')),
                macro_f1=float(f1_score(Y,pred,average='macro',zero_division=0)),
                micro_f1=float(f1_score(Y,pred,average='micro',zero_division=0)),
                macro_precision=float(precision_score(Y,pred,average='macro',zero_division=0)),
                macro_recall=float(recall_score(Y,pred,average='macro',zero_division=0)))

def label_metrics(Y,P,thr):
    pred=(P >= thr[None,:]).astype(np.int8)
    rows=[]
    for j,label in enumerate(LABELS):
        yp=Y[:,j]; pp=pred[:,j]
        rows.append(dict(label=label,positive_support=int(yp.sum()),threshold=float(thr[j]),
                         precision=float(precision_score(yp,pp,zero_division=0)),
                         recall=float(recall_score(yp,pp,zero_division=0)),
                         f1=float(f1_score(yp,pp,zero_division=0)),
                         ap=float(average_precision_score(yp,P[:,j])),
                         tp=int(((yp==1)&(pp==1)).sum()),
                         fp=int(((yp==0)&(pp==1)).sum()),
                         fn=int(((yp==1)&(pp==0)).sum())))
    return pd.DataFrame(rows)

# Unit checks on toy data BEFORE scoring any research result
_toy_y=np.array([[0,0,0,0,0,0],[1,1,1,1,1,1],[1,0,1,0,1,0]],dtype=np.int8)
_toy_p=np.array([[.1]*6,[.9]*6,[.8,.2,.8,.2,.8,.2]],dtype=float)
assert abs(metrics(_toy_y,_toy_p,np.array([.5]*6))['macro_f1']-1.0)<1e-12
assert all((select_thresholds(_toy_y,_toy_p)>=.05)&(select_thresholds(_toy_y,_toy_p)<=.95))
print('Scoring and threshold unit tests PASS.')

In [ ]:
# CELL 4 — Compare PREDECLARED fusion candidates; select by selection macro AP only
candidates={
    'FROZEN_2T': original['FROZEN_2T'],
    'R1_RoBERTa': original['R1_RoBERTa'],
    'B1_DistilBERT': original['B1_DistilBERT'],
    'A1_LR': original['A1_LR'],
    'A2_CalibratedSVM': original['A2_CalibratedSVM'],
}
# Weight w means: (1-w)*first model + w*lexical model. No fitting of model weights on test.
for w in [0.10, 0.20, 0.30]:
    for n,base,lex in [
        ('FROZEN+LR','FROZEN_2T','A1_LR'),
        ('FROZEN+SVM','FROZEN_2T','A2_CalibratedSVM'),
        ('RoBERTa+LR','R1_RoBERTa','A1_LR')
    ]:
        candidates[f'{n}_{w:.2f}'] = (1-w)*original[base] + w*original[lex]

rows=[]; per_model_thresholds={}
for name,prob in candidates.items():
    thr=select_thresholds(Yv[idx_sel],prob[idx_sel])
    per_model_thresholds[name]=thr
    selection=metrics(Yv[idx_sel],prob[idx_sel],thr)
    audit=metrics(Yv[idx_audit],prob[idx_audit],thr)
    rows.append({'candidate':name, 'selection_macro_ap':selection['macro_ap'],
                 'audit_macro_ap':audit['macro_ap'], 'audit_macro_f1':audit['macro_f1'],
                 'audit_micro_f1':audit['micro_f1'],
                 'selection_macro_f1':selection['macro_f1']})
fusion_df=pd.DataFrame(rows).sort_values('selection_macro_ap',ascending=False)
fusion_df.to_csv(OUT/'fusion_candidate_comparison.csv',index=False)
print('All scores below are DEVELOPMENT scores; the audit half was previously viewed during old model selection.')
display(fusion_df.round(4))

fusion_names=[n for n in candidates if '+' in n]
# Select strictly on selection AP (not audit AP or test metrics).
best_fusion=max(fusion_names, key=lambda x: metrics(Yv[idx_sel], candidates[x][idx_sel],per_model_thresholds[x])['macro_ap'])
selected_prob=candidates[best_fusion]
frozen_prob=candidates['FROZEN_2T']
selected_thr=per_model_thresholds[best_fusion]
frozen_thr=per_model_thresholds['FROZEN_2T']
# Retuning here is ONLY for an apples-to-apples threshold study. It does not
# replace your original frozen six-threshold policy in final_config.json.
orig_cfg=json.loads((FINAL/'final_config.json').read_text())
original_locked_thresholds=np.array([float(orig_cfg['thresholds'][l]) for l in LABELS])
locked_audit_stats=metrics(Yv[idx_audit],frozen_prob[idx_audit],original_locked_thresholds)
retuned_audit_stats=metrics(Yv[idx_audit],frozen_prob[idx_audit],frozen_thr)
print('Original frozen thresholds',dict(zip(LABELS,original_locked_thresholds)))
print('Frozen model with ORIGINAL LOCKED thresholds (audit only):',
      {k:round(v,4) for k,v in locked_audit_stats.items()})
print('Same frozen model RETUNED on selection half (audit only):',
      {k:round(v,4) for k,v in retuned_audit_stats.items()})
pd.DataFrame([dict(policy='original_locked',**locked_audit_stats),
              dict(policy='retuned_on_selection_half',**retuned_audit_stats)]).to_csv(
                  OUT/'frozen_original_vs_retuned_thresholds.csv',index=False)
sel_fusion_ap=float(fusion_df.set_index('candidate').loc[best_fusion,'selection_macro_ap'])
sel_frozen_ap=float(fusion_df.set_index('candidate').loc['FROZEN_2T','selection_macro_ap'])
print('\nBest candidate selected WITHOUT inspecting audit:',best_fusion)
print('Selection AP difference vs frozen:',round(sel_fusion_ap-sel_frozen_ap,5))
print('Audit AP difference (DESCRIPTIVE):',round(metrics(Yv[idx_audit],selected_prob[idx_audit],selected_thr)['macro_ap']-metrics(Yv[idx_audit],frozen_prob[idx_audit],frozen_thr)['macro_ap'],5))
if sel_fusion_ap-sel_frozen_ap < 0.002:
    print('CAUTION: <0.002 selection AP gain. Do NOT replace the frozen model on this evidence.')
else:
    print('Promising selection gain; check paired uncertainty and fairness before interpreting.')

p_fusion=label_metrics(Yv[idx_audit], selected_prob[idx_audit], selected_thr).assign(model=best_fusion)
p_frozen=label_metrics(Yv[idx_audit], frozen_prob[idx_audit], frozen_thr).assign(model='FROZEN_2T')
perlabel=pd.concat([p_frozen,p_fusion],ignore_index=True)
perlabel.to_csv(OUT/'fusion_per_label_audit.csv',index=False)
display(perlabel[['model','label','positive_support','precision','recall','f1','ap','tp','fp','fn']].round(4))

In [ ]:
# CELL 5 — Paired bootstrap: how uncertain is the fusion improvement?
# Both candidates are assessed on the SAME rows. Bootstraps resample comments, not independent label cells.
rng=np.random.default_rng(SEED)
Ya=Yv[idx_audit]; Pa=selected_prob[idx_audit]; Pb=frozen_prob[idx_audit]
A_thr=selected_thr; B_thr=frozen_thr
bootstrap=[]
for b in range(BOOTSTRAP_REPS):
    ix=rng.integers(0,len(Ya),len(Ya))
    ys=Ya[ix]
    # Skip replicates missing positives in any label (rare with ~12k validation rows).
    if np.any(ys.sum(axis=0)==0): continue
    ma=metrics(ys,Pa[ix],A_thr)
    mb=metrics(ys,Pb[ix],B_thr)
    bootstrap.append({'delta_macro_ap':ma['macro_ap']-mb['macro_ap'],
                      'delta_macro_f1':ma['macro_f1']-mb['macro_f1']})
boot=pd.DataFrame(bootstrap)
assert len(boot) >= BOOTSTRAP_REPS*0.9, 'Too many skipped bootstrap replicates'
boot.to_csv(OUT/'paired_bootstrap_differences.csv',index=False)
for col in boot.columns:
    lo,hi=np.quantile(boot[col],[.025,.975])
    print(f'{col}: bootstrap 95% CI [{lo:+.4f}, {hi:+.4f}]')
print('Interpretation: intervals including 0 provide insufficient evidence of a reliable improvement.')
print('Limitation: this bootstrap does NOT capture random training-seed variation, and the audit is not untouched.')

## Study C — Fairness and length sensitivity

Identity-related words are used as **imperfect text proxies**, not verified demographic attributes. A false flag here means *a ground-truth non-toxic comment scored toxic*; it does **not** prove the word caused the error. Report subgroup denominators and uncertainty; avoid claiming bias mitigation from tiny changes.

In [ ]:
# CELL 6 — Paired subgroup false-positive audit, same audit comments
IDENTITY_TERMS=['gay','lesbian','homosexual','muslim','jewish','christian','black','white','woman']
audit_text=val.iloc[idx_audit].comment_text.fillna('').astype(str)
toxic_label_index=LABELS.index('toxic')
negative=(Ya[:,toxic_label_index]==0)
allneg=int(negative.sum())

def subgroup_fpr(P,thr):
    return ((P[:,0] >= thr[0]) & negative)
false_a=subgroup_fpr(Pa,A_thr)
false_b=subgroup_fpr(Pb,B_thr)
rows=[]
all_mask=np.zeros(len(audit_text),dtype=bool)
for term in IDENTITY_TERMS:
    has=audit_text.str.contains(r'\b'+re.escape(term)+r'\b',case=False,regex=True).to_numpy()
    all_mask|=has
    g=has&negative
    n=int(g.sum())
    if n==0: continue
    rows.append({'term':term,'n_non_toxic':n,
                 'frozen_fp':int((false_b&g).sum()),
                 'fusion_fp':int((false_a&g).sum()),
                 'frozen_fpr':float((false_b&g).sum()/n),
                 'fusion_fpr':float((false_a&g).sum()/n),
                 'delta_fpr':float(((false_a&g).sum()-(false_b&g).sum())/n),
                 'support_warning':'LOW SUPPORT (<30)' if n<30 else ''})
# Aggregate ANY term (overlap counted once)
g=all_mask&negative; n=int(g.sum())
if n:
    rows.append({'term':'ANY_IDENTITY_PROXY','n_non_toxic':n,
                 'frozen_fp':int((false_b&g).sum()),'fusion_fp':int((false_a&g).sum()),
                 'frozen_fpr':float((false_b&g).sum()/n),
                 'fusion_fpr':float((false_a&g).sum()/n),
                 'delta_fpr':float(((false_a&g).sum()-(false_b&g).sum())/n),
                 'support_warning':'LOW SUPPORT (<30)' if n<30 else ''})
fair=pd.DataFrame(rows)
fair.to_csv(OUT/'identity_proxy_false_positive_audit.csv',index=False)
print(f'Overall ground-truth-non-toxic audit comments: {allneg}')
print('Overall frozen toxic FPR:',round(float(false_b.sum()/allneg),4))
print('Overall fusion toxic FPR:',round(float(false_a.sum()/allneg),4))
display(fair.round(4))
print('IMPORTANT: These are simple keyword-proxy subgroup statistics, NOT causal identity-bias measurements.')

In [ ]:
# CELL 7 — Noncausal length-stress analysis, same audit comments
words=audit_text.str.split().map(len).to_numpy()
bins=[('SHORT_<=40_words',words<=40),('MEDIUM_41_to_120',(words>40)&(words<=120)),
      ('LONG_>120_words',words>120)]
length_rows=[]
for label,mask in bins:
    if int(mask.sum())<30: continue
    for name,PP,TT in [('FROZEN_2T',Pb,B_thr),(best_fusion,Pa,A_thr)]:
        yy=Ya[mask];pp=PP[mask]
        rr=metrics(yy,pp,TT)
        pos=int(yy[:,0].sum())
        toxic_recall=(int(((pp[:,0]>=TT[0])&(yy[:,0]==1)).sum())/pos) if pos else np.nan
        # Macro AP valid only if every label has at least one positive in the subset
        length_rows.append({'length_bin':label,'model':name,'n':len(yy),
                            'toxic_positive_support':pos,'toxic_recall':toxic_recall,
                            'macro_f1':rr['macro_f1'],
                            'macro_ap':rr['macro_ap'] if np.all(yy.sum(axis=0)>0) else np.nan})
length_df=pd.DataFrame(length_rows)
length_df.to_csv(OUT/'length_stress_audit.csv',index=False)
display(length_df.round(4))
print('Word count is only a LENGTH PROXY; do not claim this isolates transformer token truncation.')

## Study D — Controlled 34K enriched vs representative 34K vs full 111K

**Why this adds genuinely new value to your old work:** your prior transformers and LR were trained on 34,044 comments containing **all 11,348 positive-any-label training comments**, plus 22,696 negatives. That changes prevalence. This experiment adds a *representative* 34K control, to distinguish **enrichment** from **training size**.

| Condition | N | Sampling | Class weights |
|---|---:|---|---|
| `ENRICHED_34K` | 34,044 | Your actual old `train_sub.csv` | none |
| `REPRESENTATIVE_34K` | 34,044 | Six-label-pattern-stratified subset of `train_full.csv` | none |
| `FULL_111K` | 111,707 | Your full saved training partition | none |
| `FULL_111K_BALANCED` *(optional)* | 111,707 | Same full training partition | balanced per binary label |

**Controls:** same train-only fitted word TF-IDF vocabulary, same SGD-logistic loss/regularization/iteration budget, same random seed, same six labels, same selection/audit evaluation procedure. This is a **new controlled word-TF-IDF experiment**, not a re-run of old A1, so do not compare its numeric AP as though vectorization settings were identical to historical A1. Here we deliberately use a memory-bounded **60K word unigram/bigram TF-IDF** feature space. Optional character features can be enabled later, but are off by default to keep this study fast and avoid Colab memory spikes.

**Scope:** This demonstrates the importance of training-data composition with a *classical model*. It cannot by itself prove that full-data DistilBERT or RoBERTa will behave the same way. That is a later GPU study.

In [ ]:
# CELL 8 — Create genuinely comparable subsets and document prevalence
if RUN_SAMPLING_STUDY:
    nsub=len(sub)
    keep,_ = stratified_multilabel_indices(Ytr, train_size=nsub, seed=SEED)
    rep=tr.iloc[keep].copy().reset_index(drop=True)
    assert len(rep)==nsub and rep.id.is_unique
    labels_by_split=[]
    for nm,dd in [('ENRICHED_34K',sub),('REPRESENTATIVE_34K',rep),('FULL_111K',tr)]:
        yy=dd[LABELS].to_numpy()
        row={'condition':nm,'n':len(dd),'any_positive_fraction':float((yy.sum(axis=1)>0).mean())}
        row.update({f'{lbl}_prevalence':float(yy[:,j].mean()) for j,lbl in enumerate(LABELS)})
        labels_by_split.append(row)
    prevalence=pd.DataFrame(labels_by_split)
    prevalence.to_csv(OUT/'training_prevalence_comparison.csv',index=False)
    display(prevalence.round(4))
    print('Representative 34K is six-label-pattern-stratified from TRAIN only; neither validation nor test is touched.')
else:
    print('Study D disabled; A–C results remain valid.')

In [ ]:
# CELL 9 — Train controlled lexical models with a shared train-only representation
# Colab CPU is sufficient; allocating TF-IDF for 111K comments may take several minutes.
if RUN_SAMPLING_STUDY:
    import scipy.sparse as sp
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import SGDClassifier
    from sklearn.multiclass import OneVsRestClassifier
    import joblib

    word_vec=TfidfVectorizer(analyzer='word',ngram_range=(1,2),max_features=60000,
                             min_df=3,sublinear_tf=True,dtype=np.float32)
    INCLUDE_CHAR_FEATURES=False  # OFF by default for fast, memory-safe training-data ablation
    # Turning this on can take considerably longer; do not change it mid-comparison.
    char_vec=TfidfVectorizer(analyzer='char',ngram_range=(3,5),max_features=30000,
                             min_df=3,sublinear_tf=True,dtype=np.float32) if INCLUDE_CHAR_FEATURES else None
    text_full=tr.comment_text.fillna('').astype(str).tolist()
    print('Fitting vocabulary on ALL TRAINING TEXT only. This is shared, unsupervised vocabulary.')
    t0=time.time()
    Xtrain=word_vec.fit_transform(text_full).tocsr()
    text_val=val.comment_text.fillna('').astype(str).tolist()
    Xval=word_vec.transform(text_val).tocsr()
    if INCLUDE_CHAR_FEATURES:
        Xtrain=sp.hstack((Xtrain,char_vec.fit_transform(text_full)),format='csr',dtype=np.float32)
        Xval=sp.hstack((Xval,char_vec.transform(text_val)),format='csr',dtype=np.float32)
    print('Shared training features:',Xtrain.shape,'| minutes:',round((time.time()-t0)/60,2))
    assert Xtrain.shape[1]==Xval.shape[1]

    full_index=pd.Series(np.arange(len(tr)),index=tr.id.astype(str))
    datasets={
        'ENRICHED_34K':sub,
        'REPRESENTATIVE_34K':rep,
        'FULL_111K':tr,
    }
    if RUN_BALANCED_VARIANT:
        datasets['FULL_111K_BALANCED']=tr
    study_rows=[]; sampling_pred={}; sampling_thresholds={}
    for name,dd in datasets.items():
        tic=time.time()
        ids=dd.id.astype(str).to_numpy()
        row_ix=full_index.loc[ids].to_numpy()
        yy=dd[LABELS].to_numpy(dtype=np.int8)
        balanced=name.endswith('BALANCED')
        # Efficient linear logistic classifier. It minimizes logistic loss using SGD,
        # so this experiment is NOT directly comparable in absolute score to old liblinear A1.
        learner=SGDClassifier(loss='log_loss', alpha=1e-5, max_iter=25,
                              tol=1e-3, random_state=42, average=True,
                              class_weight='balanced' if balanced else None)
        model=OneVsRestClassifier(learner,n_jobs=1)
        print('\nTraining',name,'| n=',len(dd),'| balanced=',balanced)
        model.fit(Xtrain[row_ix],yy)
        scores=np.asarray(model.predict_proba(Xval),dtype=np.float32)
        assert scores.shape==(len(val),6) and np.isfinite(scores).all()
        assert (scores>=0).all() and (scores<=1).all()
        sampling_pred[name]=scores
        thr=select_thresholds(Yv[idx_sel],scores[idx_sel]); sampling_thresholds[name]=thr
        sel=metrics(Yv[idx_sel],scores[idx_sel],thr)
        audit=metrics(Yv[idx_audit],scores[idx_audit],thr)
        np.save(OUT/f'{name}_val_probs.npy',scores)
        study_rows.append({'model':name,'training_n':len(dd),
                           'training_positive_rate':float((yy.sum(axis=1)>0).mean()),
                           'selection_macro_ap':sel['macro_ap'],
                           'audit_macro_ap':audit['macro_ap'],
                           'audit_macro_f1':audit['macro_f1'],
                           'audit_micro_f1':audit['micro_f1'],
                           'minutes':(time.time()-tic)/60})
        if SAVE_NEW_LEXICAL_MODELS:
            joblib.dump(model,OUT/f'{name}_SGD_logistic_model.joblib',compress=3)
        print('   selection AP',round(sel['macro_ap'],4),'| audit AP',round(audit['macro_ap'],4),
              '| audit F1',round(audit['macro_f1'],4),'| minutes',round((time.time()-tic)/60,2))
    if SAVE_NEW_LEXICAL_MODELS:
        joblib.dump({'word_vectorizer':word_vec,'char_vectorizer':char_vec,
                     'label_order':LABELS},OUT/'SHARED_train_only_vectorizers.joblib',compress=3)
    sample_table=pd.DataFrame(study_rows).sort_values('selection_macro_ap',ascending=False)
    sample_table.to_csv(OUT/'sampling_study_scores.csv',index=False)
    (OUT/'sampling_thresholds.json').write_text(json.dumps({k:dict(zip(LABELS,v.tolist())) for k,v in sampling_thresholds.items()},indent=2))
    display(sample_table.round(4))

In [ ]:
# CELL 10 — Analyze the controlled sampling study and its per-class errors
if RUN_SAMPLING_STUDY:
    details=[]
    for nm,P in sampling_pred.items():
        details.append(label_metrics(Yv[idx_audit],P[idx_audit],sampling_thresholds[nm]).assign(model=nm))
    study_perlabel=pd.concat(details,ignore_index=True)
    study_perlabel.to_csv(OUT/'sampling_per_label_audit.csv',index=False)
    display(study_perlabel[['model','label','positive_support','precision','recall','f1','ap','tp','fp','fn']].round(4))
    # One transparent subgroup check on the SAME audit half, without changing thresholds.
    Ya=Yv[idx_audit]
    audit_text=val.iloc[idx_audit].comment_text.fillna('').astype(str)
    neg=Ya[:,0]==0
    proxy=audit_text.str.contains(r'\b(?:gay|lesbian|homosexual|muslim|jewish|christian)\b',
                                  case=False,regex=True).to_numpy() & neg
    print('N ground-truth non-toxic identity-keyword audit comments:',int(proxy.sum()))
    if proxy.sum()>=20:
        fpr_rows=[]
        for nm,P in sampling_pred.items():
            p=P[idx_audit]
            fp=(p[:,0]>=sampling_thresholds[nm][0]) & neg
            fpr_rows.append({'model':nm,'n_non_toxic_keyword':int(proxy.sum()),
                             'keyword_fpr':float(fp[proxy].mean()),'overall_nontoxic_fpr':float(fp[neg].mean())})
        fair_sampling=pd.DataFrame(fpr_rows)
        fair_sampling.to_csv(OUT/'sampling_keyword_fpr.csv',index=False)
        display(fair_sampling.round(4))
    else:
        print('Too little subgroup support for useful estimates; do NOT infer fairness improvement.')

## Final cell — Export findings and write a responsible conclusion

The notebook deliberately **does not replace** your existing frozen ensemble. A worthwhile research result can be: “fusion brought no reliable benefit,” or “representative training reduced false positives but sacrificed some rare-class recall.” Document both tradeoffs with the actual printed values.

**What to show in your mid-evaluation:** (1) data-composition prevalence table, (2) paired fusion comparison and uncertainty, (3) per-label F1 and identity-proxy false positives. This is evidence of research reasoning rather than another model name.

In [ ]:
# CELL 11 — Final report/manifest; no official test touched, original results unchanged
summary_lines=[
    '# Phase 2 Value-Adding Studies — Research Notes',
    '',
    'All numbers below are from original development validation records. The original models had already',
    'been selected using these records, so the retrospective audit is not an independent final test.',
    '',
    '## Fusion experiment',
    f'- Best predeclared fusion on selection macro AP: **{best_fusion}**',
    f'- Selection macro AP: fusion {sel_fusion_ap:.4f} vs frozen {sel_frozen_ap:.4f}',
    f'- Audit macro AP: fusion {metrics(Ya,Pa,A_thr)["macro_ap"]:.4f} vs frozen {metrics(Ya,Pb,B_thr)["macro_ap"]:.4f}',
    f'- Audit macro F1: fusion {metrics(Ya,Pa,A_thr)["macro_f1"]:.4f} vs frozen {metrics(Ya,Pb,B_thr)["macro_f1"]:.4f}',
    '- Bootstrap CIs are in paired_bootstrap_differences.csv. Do not declare a clear winner if interval crosses 0.',
    '',
    '## Fairness and stress diagnostics',
    '- See identity_proxy_false_positive_audit.csv and length_stress_audit.csv.',
    '- Identity keywords are imperfect proxies; a keyword subgroup difference is not proof of causation.',
]
if RUN_SAMPLING_STUDY:
    summary_lines += [
        '', '## Controlled lexical training-data ablation',
        '- All three main SGD-logistic variants (plus optional balanced version) use a shared train-only word TF-IDF vocabulary (character option off by default).',
        '- Same features, model family, validation records, and threshold-tuning method.',
        '- Main results: sampling_study_scores.csv and sampling_per_label_audit.csv.',
        '- This is an SGD-logistic result; transformer impact still requires a controlled GPU study.',
    ]
summary_lines += [
    '', '## Next priority',
    '1. Keep the frozen model, its saved thresholds, 20 LIME explanations and past test results.',
    '2. Add these new controlled validation experiments to the evidence section.',
    '3. Only if justified: train a full-data DistilBERT BCE experiment with a new ID on the SAME old split.',
    '4. Do not repeatedly evaluate new candidates on the previously inspected official test.',
    '5. Complete manual error analysis, fairness interpretation and deployment brief.',
    '', '## Explicit scope',
    '- No retraining of RoBERTa/DistilBERT, no changes to saved frozen thresholds, no final-test evaluation. For apples-to-apples fusion comparisons the old model scores get thresholds re-tuned on the same selection half; original locked thresholds are also reported separately.',
]
(OUT/'PHASE2_RESEARCH_NOTES.md').write_text('\n'.join(summary_lines),encoding='utf-8')
print('\n'.join(summary_lines))
print('\nAll new files saved in:',OUT)
print('Your original notebooks, split CSVs, model predictions, and frozen configuration are UNCHANGED.')